### Day 19: 一些数据可视化

Bonjour Sarah! 今天让我们运用之前学的知识到实际的生活中。以下情景是Ken突然想到的

今天早上 Ken 迷迷糊糊醒来，发现 eeg 信号很奇怪，大范围（几伏）抖动，用手摸了一下线，似乎又好了。他怀疑线有问题，但是似乎也复刻不出来原先的大范围抖动了。为了让接下来的人（ Sarah 和 Sipeng ）不再出现这个问题，也为了确保昨晚的数据是正常的，Ken决定用python来可视化一下。

Ken 把保存好的 gyroscope 和 channel 6 的 eeg 数据放在了这个文件夹里。接下来由Sarah自由发挥，看看Sarah能不能测算出：

 - Ken 的 eeg 信号在什么时间内抖动？昨晚的数据是否大部分都不可用？
 - 根据 gyroscope 的数据，Ken什么时候在睡觉？

听起来挺有意思的，让我们开始吧哈哈哈。我会先自己写一遍作为参考答案，然后把我的代码扣掉，交给Sarah做练习

In [ ]:
# 首先计算一下数据会不会让内存爆掉
# eeg 数据 1KHz，八小时计算的话，假设使用float64，那么需要 8(h) * 60(min/h) * 60(s/min) * 1000(sample/s) * 8(byte/sample) = 230400000 byte = 230.4 MB
# gyroscope 的数据只有 100Hz，那就更存得下了！

In [ ]:
# 读取第一个文件，并将 acc_x, acc_y, acc_z 合成一个向量，并将其 plot 出来

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

acc_x_path = r"20250706_10\Nora Intan RHD ICM.acc_x\0001.csv"
acc_y_path = r"20250706_10\Nora Intan RHD ICM.acc_y\0001.csv"
acc_z_path = r"20250706_10\Nora Intan RHD ICM.acc_z\0001.csv"

df_x = pd.read_csv(acc_x_path, header=None, names=['time', 'acc_x'])
df_y = pd.read_csv(acc_y_path, header=None, names=['time', 'acc_y'])
df_z = pd.read_csv(acc_z_path, header=None, names=['time', 'acc_z'])
# df = pd.concat([df_x, df_y, df_z], axis=1)
df = df_x.merge(df_y, on='time', how='inner').merge(df_z, on='time', how='inner')  # 这也是跟 GPT 学的

df['acc_mag'] = np.sqrt(df['acc_x']**2 + df['acc_y']**2 + df['acc_z']**2)

plt.plot(df['time'], df['acc_mag'])
plt.show()

In [ ]:
# 读取所有acc文件，并将其合成一个向量，并将其 plot 出来

df_list = []

for index in range(1, 62):
    pass

In [ ]:
# 检查合成后的 df 是否有 NaN（Not a Number）值

Ken 的吐槽：啊哈哈哈还是很乱。看来我睡觉总是翻身打滚。

哦对了，咱们可以低通滤波。

以下是GPT写的，看看就好了，我也不会写。以后可以学一学

In [ ]:
# 对acc_mag数据进行低通滤波，以减少高频噪声
from scipy import signal
import matplotlib

# 设置中文字体
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei']
matplotlib.rcParams['axes.unicode_minus'] = False


# 假设采样频率为100Hz（根据数据结构推测）
fs = 100  # 采样频率
cutoff = 0.3  # 截止频率0.3Hz，用于去除高频运动噪声

# 设计巴特沃斯低通滤波器
nyquist = fs / 2  # 奈奎斯特频率
normal_cutoff = cutoff / nyquist
b, a = signal.butter(4, normal_cutoff, btype='low', analog=False)

# 应用滤波器
df_all['acc_mag_filtered'] = signal.filtfilt(b, a, df_all['acc_mag'])

# 绘制原始信号和滤波后的信号对比
plt.figure(figsize=(15, 10))

# 第一个子图：两个信号叠加显示
plt.subplot(3, 1, 1)
plt.plot(df_all['time'], df_all['acc_mag'], alpha=0.6, label='原始信号', color='blue')
plt.plot(df_all['time'], df_all['acc_mag_filtered'], label='低通滤波后', color='red', linewidth=2)
plt.title('原始信号 vs 低通滤波后信号对比')
plt.ylabel('加速度 (g)')
plt.legend()
plt.grid(True)

# 第二个子图：只显示原始信号
plt.subplot(3, 1, 2)
plt.plot(df_all['time'], df_all['acc_mag'], alpha=0.7, label='原始信号', color='blue')
plt.title('原始加速度幅值')
plt.ylabel('加速度 (g)')
plt.legend()
plt.grid(True)

# 第三个子图：只显示滤波后信号
plt.subplot(3, 1, 3)
plt.plot(df_all['time'], df_all['acc_mag_filtered'], 'r-', label='低通滤波后', linewidth=2)
plt.title('低通滤波后的加速度幅值')
plt.xlabel('时间 (s)')
plt.ylabel('加速度 (g)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# 计算滤波后数据的变化程度来判断运动状态
df_all['acc_variation'] = df_all['acc_mag_filtered'].rolling(window=500, center=True).std()

plt.figure(figsize=(12, 6))
plt.plot(df_all['time'], df_all['acc_variation'])
plt.title('加速度变化程度 (滑动标准差) - 可用于判断睡眠状态')
plt.xlabel('时间 (s)')
plt.ylabel('加速度变化程度')
plt.grid(True)
plt.show()


Ken 再次吐槽：完全看不出来端倪啊怎么回事

哦对，这些数据除了一开始的一点点我应该是全程在睡觉，所以每次一翻身/晃脑袋就会导致加速度有一次变化。如果我是醒着的，加速度变化程度应该会比这个乱很多很多

那让我们看一看eeg的数据有没有异常吧

In [ ]:
# 查看第一个文件，并将其 plot 出来

import matplotlib
from matplotlib import pyplot as plt
import pandas as pd

# 设置中文字体
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei']
matplotlib.rcParams['axes.unicode_minus'] = False

file_path = r"20250706_10\Nora Intan RHD ICM.intan_ch6\0001.csv"

df = pd.read_csv(file_path, header=None, names=['time', 'eeg'])

plt.plot(df['time'], df['eeg'])
plt.show()

In [ ]:
# 加载所有文件

df_list = []

# for ...

df_all = None  # 请你自己写一下
df_all

In [ ]:
# plot 所有文件
# note 如果不想给电脑太大压力，可以下采样，即每隔几个数据点取一个点

df_sampled = df_all.iloc[::100]

plt.figure(figsize=(15, 6))
plt.plot(df_sampled['time'], df_sampled['eeg'])
plt.title('完整EEG数据')
plt.xlabel('时间 (s)')
plt.ylabel('电压 (V)')
plt.show()

In [ ]:
# 只 plot -0.5 - 1 v 的区间

plt.figure(figsize=(15, 6))
plt.plot(df_sampled['time'], df_sampled['eeg'])
plt.ylim(-0.5, 1)
plt.title('EEG数据 (放大视图: -0.5V 到 1V)')
plt.xlabel('时间 (s)')
plt.ylabel('电压 (V)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 分析EEG数据中的异常 - 寻找Ken怀疑的大幅度抖动
# 计算EEG信号的统计特征来识别异常

# 计算滑动窗口的标准差来检测信号的变异性
window_size = 1000  # 窗口大小
df_sampled['eeg_std'] = df_sampled['eeg'].rolling(window=window_size, center=True).std()

# 计算信号的绝对值
df_sampled['eeg_abs'] = df_sampled['eeg'].abs()

# 设置异常检测阈值
eeg_threshold = df_sampled['eeg'].std() * 3  # 3倍标准差作为异常阈值
std_threshold = df_sampled['eeg_std'].quantile(0.95)  # 95分位数作为变异性阈值

print(f"EEG信号统计:")
print(f"均值: {df_sampled['eeg'].mean():.4f} V")
print(f"标准差: {df_sampled['eeg'].std():.4f} V") 
print(f"最大值: {df_sampled['eeg'].max():.4f} V")
print(f"最小值: {df_sampled['eeg'].min():.4f} V")
print(f"异常阈值: ±{eeg_threshold:.4f} V")

# 找出可能的异常时间段
abnormal_points = df_sampled[df_sampled['eeg_abs'] > eeg_threshold]
high_variance_points = df_sampled[df_sampled['eeg_std'] > std_threshold]

print(f"\n检测到的异常:")
print(f"超过阈值的数据点: {len(abnormal_points)} 个")
print(f"高变异性数据点: {len(high_variance_points)} 个")

if len(abnormal_points) > 0:
    print(f"异常时间范围: {abnormal_points['time'].min():.1f}s - {abnormal_points['time'].max():.1f}s")

# 可视化异常检测结果
plt.figure(figsize=(15, 12))

# 子图1: EEG信号 + 异常点标记
plt.subplot(3, 1, 1)
plt.plot(df_sampled['time'], df_sampled['eeg'], 'b-', alpha=0.7, linewidth=0.5)
if len(abnormal_points) > 0:
    plt.scatter(abnormal_points['time'], abnormal_points['eeg'], color='red', s=1, alpha=0.8)
plt.axhline(y=eeg_threshold, color='r', linestyle='--', alpha=0.5, label=f'异常阈值: ±{eeg_threshold:.3f}V')
plt.axhline(y=-eeg_threshold, color='r', linestyle='--', alpha=0.5)
plt.title('EEG信号 + 异常检测')
plt.ylabel('电压 (V)')
plt.legend()
plt.grid(True, alpha=0.3)

# 子图2: 信号变异性
plt.subplot(3, 1, 2)
plt.plot(df_sampled['time'], df_sampled['eeg_std'], 'g-', linewidth=1)
if len(high_variance_points) > 0:
    plt.scatter(high_variance_points['time'], high_variance_points['eeg_std'], color='orange', s=1)
plt.axhline(y=std_threshold, color='orange', linestyle='--', label=f'高变异阈值: {std_threshold:.4f}')
plt.title('EEG信号变异性 (滑动标准差)')
plt.ylabel('标准差 (V)')
plt.legend()
plt.grid(True, alpha=0.3)

# 子图3: 放大异常区域 (如果有异常的话)
plt.subplot(3, 1, 3)
if len(abnormal_points) > 0:
    # 找到异常时间段并放大显示
    time_margin = 100  # 前后扩展100秒
    start_time = max(abnormal_points['time'].min() - time_margin, df_sampled['time'].min())
    end_time = min(abnormal_points['time'].max() + time_margin, df_sampled['time'].max())
    
    mask = (df_sampled['time'] >= start_time) & (df_sampled['time'] <= end_time)
    zoom_data = df_sampled[mask]
    
    plt.plot(zoom_data['time'], zoom_data['eeg'], 'b-', linewidth=1)
    abnormal_zoom = abnormal_points[(abnormal_points['time'] >= start_time) & (abnormal_points['time'] <= end_time)]
    if len(abnormal_zoom) > 0:
        plt.scatter(abnormal_zoom['time'], abnormal_zoom['eeg'], color='red', s=10)
    plt.title(f'异常区域放大视图 ({start_time:.1f}s - {end_time:.1f}s)')
else:
    plt.plot(df_sampled['time'], df_sampled['eeg'], 'b-', linewidth=0.5)
    plt.title('未检测到明显异常 - 完整EEG信号')

plt.xlabel('时间 (s)')
plt.ylabel('电压 (V)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


Ken 的评价：真是完了。这一次的作业出的很不好，而且这个eeg信号是什么鬼！为什么有这么多异常值？！是不是需要叫 Savanna 修一下了……

算了就这样吧 QAQ

无论如何这是一次实战的尝试。希望能让 Sarah 同学有所收获。